In [ ]:
import run_centralforce_sindy
import get_planetary_data
import time
import csv
import os
import warnings
import numpy as np

# ── Sweep configuration ──────────────────────────────────────────────────────
YEARS_VALUES          = [5, 10, 15, 25, 40, 50, 80, 100]          # number of years
INV_POINTS_PER_YEAR   = [1/10, 1/20, 1/50, 1/100, 1/250, 1/500, 1/750] # 1 / (data points per year)

# Bodies: 0=Sun, 1=Mercury, 2=Venus, 3=Earth, 4=Mars, 5=Jupiter
BODY_NAMES = ["Sun", "Mercury", "Venus", "Earth", "Mars", "Jupiter"]

# For each body, the list of *other* body names (i.e. the columns of its row)
# Body i is excluded, so the other bodies are all bodies except index i.
OTHER_BODIES = {
    i: [BODY_NAMES[j] for j in range(len(BODY_NAMES)) if j != i]
    for i in range(len(BODY_NAMES))
}

OUTPUT_CSV = "parameter_sweep_results.csv"

# ── CSV header ────────────────────────────────────────────────────────────────
# We record: years, inv_points_per_year, time_taken,
# then for each body (solver), the Sun mass estimate and Jupiter mass estimate.
header = ["years", "inv_points_per_year", "time_taken_s"]
for body_idx, body_name in enumerate(BODY_NAMES):
    others = OTHER_BODIES[body_idx]
    if "Sun" in others:
        header.append(f"{body_name}_solver_sun_mass")
    if "Jupiter" in others:
        header.append(f"{body_name}_solver_jupiter_mass")


def extract_mass(sindy_result_row):
    """
    sindy_result_row is [[m1, m2, m3, m4, m5]] — a single array of all
    mass estimates for the other bodies, in order.
    """
    arr = np.array(sindy_result_row).flatten()
    return arr.tolist()


def run_sweep():
    os.makedirs("pickles", exist_ok=True)

    with open(OUTPUT_CSV, "w", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=header)
        writer.writeheader()

        total = len(YEARS_VALUES) * len(INV_POINTS_PER_YEAR)
        run_num = 0

        for years in YEARS_VALUES:
            for inv_pts in INV_POINTS_PER_YEAR:
                run_num += 1
                pts_label = round(1 / inv_pts)
                pickle_name = f"pickles/allPlanets_{years}years_{pts_label}per.pickle"

                print(f"\n{'='*60}")
                print(f"Run {run_num}/{total}: years={years}, "
                      f"inv_points_per_year={inv_pts:.4f} "
                      f"(~{pts_label} pts/yr)")
                print(f"{'='*60}")

                # ── Extract planetary data ────────────────────────────────
                print("Extracting planetary data...")
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    get_planetary_data.extract_multiplanetary_data(
                        years,
                        inv_pts,
                        doPickle=True,
                        pickleFilename=pickle_name
                    )

                # ── Run SINDy & time it ───────────────────────────────────
                print("Running SINDy...")
                t_start = time.perf_counter()
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    c = run_centralforce_sindy.run_centralforce_sindy(
                        pickle_name,
                        fileFormat="pickle",
                        excludeCols=[6],
                        doMerged=True,
                        doUnmerged=False,
                        minPolynomialOrder=2,
                        maxPolynomialOrder=2
                    )
                t_end = time.perf_counter()
                elapsed = t_end - t_start
                print(f"SINDy finished in {elapsed:.2f}s")

                # ── Parse results ─────────────────────────────────────────
                row = {
                    "years": years,
                    "inv_points_per_year": inv_pts,
                    "time_taken_s": round(elapsed, 4),
                }

                for body_idx, body_rows in enumerate(c):
                    body_name  = BODY_NAMES[body_idx]
                    others     = OTHER_BODIES[body_idx]
                    masses     = extract_mass(body_rows)

                    mass_dict  = dict(zip(others, masses))

                    sun_key     = f"{body_name}_solver_sun_mass"
                    jupiter_key = f"{body_name}_solver_jupiter_mass"

                    if sun_key in header:
                        row[sun_key]     = mass_dict.get("Sun", float("nan"))
                    if jupiter_key in header:
                        row[jupiter_key] = mass_dict.get("Jupiter", float("nan"))

                    # Pretty-print for the terminal
                    print(f"  {body_name}: ", end="")
                    for other, mass in mass_dict.items():
                        print(f"{other}={mass:.4e}  ", end="")
                    print()

                writer.writerow(row)
                csvfile.flush()   # write incrementally in case of crash

    print(f"\nSweep complete. Results saved to '{OUTPUT_CSV}'.")


if __name__ == "__main__":
    run_sweep()


Run 1/63: years=5, inv_points_per_year=0.1000 (~10 pts/yr)
Extracting planetary data...
Running SINDy...
[18, 19, 20]
SINDy finished in 0.02s
  Sun: Mercury=1.5015e-07  Venus=-1.7517e-03  Earth=-2.6741e-03  Mars=-7.9147e-04  Jupiter=-9.0269e-01  
  Mercury: Sun=-1.5073e+01  Venus=2.3265e+01  Earth=1.8700e+01  Mars=-2.5035e+02  Jupiter=-4.7480e+02  
  Venus: Sun=-6.9344e+02  Mercury=-1.0464e+01  Earth=-1.3131e+00  Mars=2.0813e+01  Jupiter=1.0109e+03  
  Earth: Sun=-8.9201e+02  Mercury=-6.8518e+00  Venus=1.2770e+00  Mars=-1.3966e+01  Jupiter=7.3476e+01  
  Mars: Sun=-9.5731e+02  Mercury=-1.3486e+01  Venus=-4.9587e+00  Earth=-9.9805e+00  Jupiter=-1.5937e+02  
  Jupiter: Sun=-1.0290e+03  Mercury=4.7868e+00  Venus=-3.4185e+00  Earth=2.5312e-01  Mars=8.8415e-01  

Run 2/63: years=5, inv_points_per_year=0.0500 (~20 pts/yr)
Extracting planetary data...
Running SINDy...
[18, 19, 20]
SINDy finished in 0.02s
  Sun: Mercury=-4.5557e-05  Venus=-2.2811e-03  Earth=-2.9444e-03  Mars=-8.6762e-04  Jupi

KeyboardInterrupt: 